# ArcGIS Publishing Automation for MPC Pro

This notebook automates the workflow of discovering new imagery from a Microsoft Planetary Computer (MPC) Pro GeoCatalog and publishing it to ArcGIS Enterprise as an imagery layer via Image Server.

## Workflow
1. **Configure** — Set the STAC query, image collection, and connection parameters
2. **Authenticate** — Connect to MPC Pro GeoCatalog and ArcGIS Enterprise
3. **Discover** — Query the GeoCatalog for new STAC items
4. **Deduplicate** — Check image collection for already-ingested items
5. **Ingest** — Add new COGs to the image collection via cloud store paths
6. **Publish** — Publish or refresh the imagery layer on Image Server

## Prerequisites
- MPC Pro GeoCatalog with imagery collections
- ArcGIS Enterprise with Image Server
- Azure Blob Storage registered as a cloud data store in ArcGIS Enterprise
- Managed identity (or service principal) with **GeoCatalog Reader** role

## Scheduling
To run this notebook on a schedule, use ArcGIS Notebook Server's built-in
**Schedule a notebook task** feature (available in the portal's Notebook Manager).

## 1. Configuration

Edit the configuration below to match your environment.

In [ ]:
# =============================================================================
# Configuration
# =============================================================================

config = {
    # MPC Pro GeoCatalog
    "geocatalog": {
        "endpoint": "https://your-geocatalog.geocatalog.spatio.azure.com",
        "api_version": "2025-04-30-preview",
    },
    
    # STAC Query Parameters
    "stac_query": {
        "collections": ["sentinel-2-l2a"],
        "bbox": [-122.5, 37.5, -122.0, 38.0],  # San Francisco Bay Area
        "datetime": "2024-01-01T00:00:00Z/2024-12-31T23:59:59Z",
        "filter": {"op": "<", "args": [{"property": "eo:cloud_cover"}, 20]},
        "filter_lang": "cql2-json",
        "limit": 100,
    },
    
    # Image Collection
    "image_collection": {
        "name": "sentinel2_imagery",
        "description": "Sentinel-2 L2A imagery from MPC Pro GeoCatalog",
        "coordinate_system": 4326,
        "raster_type_name": "Raster Dataset",
        "create_if_missing": True,
        "source_asset_key": "visual",
    },
    
    # ArcGIS Enterprise Connection
    "arcgis_enterprise": {
        "portal_url": "https://your-portal.domain.com/portal",
        "image_server_url": "https://your-server.domain.com/server",
        "auth_method": "username_password",  # or "oauth", "windows"
        "username": "",  # Leave empty to be prompted
        "password": "",  # Leave empty to be prompted
    },
    
    # Cloud Data Store (must be pre-registered in ArcGIS Enterprise)
    "cloud_store": {
        "store_name": "mpc_imagery_store",
        "storage_account": "your_storage_account",
        "container": "your_container",
    },
}

print("Configuration loaded.")
print(f"  GeoCatalog:     {config['geocatalog']['endpoint']}")
print(f"  Collections:    {config['stac_query']['collections']}")
print(f"  Image Collection: {config['image_collection']['name']}")
print(f"  Portal:         {config['arcgis_enterprise']['portal_url']}")

## 2. Install and Import Dependencies

In [ ]:
# Install dependencies (uncomment if needed)
# !pip install azure-planetarycomputer azure-identity arcgis pyyaml

import logging
from datetime import datetime
from urllib.parse import urlparse

from azure.identity import DefaultAzureCredential
from azure.planetarycomputer import PlanetaryComputerProClient
from azure.planetarycomputer.models import StacSearchParameters
from arcgis.gis import GIS
from arcgis.raster import ImageryLayer
from arcgis.raster.analytics import create_image_collection

# Configure logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger("publishing_automation")

print("All dependencies imported successfully.")

## 3. Authenticate

Connect to both MPC Pro GeoCatalog (via Azure AD) and ArcGIS Enterprise.

In [ ]:
# --- Authenticate to MPC Pro GeoCatalog ---
# DefaultAzureCredential will use:
#   - Managed Identity (if running on Azure VM / Notebook Server with MI)
#   - Azure CLI credentials (if az login has been run)
#   - Environment variables (AZURE_CLIENT_ID, AZURE_TENANT_ID, AZURE_CLIENT_SECRET)

credential = DefaultAzureCredential()
mpc_client = PlanetaryComputerProClient(
    endpoint=config["geocatalog"]["endpoint"],
    credential=credential,
)
print(f"Connected to MPC Pro GeoCatalog: {config['geocatalog']['endpoint']}")

# --- Authenticate to ArcGIS Enterprise ---
enterprise = config["arcgis_enterprise"]
username = enterprise["username"]
password = enterprise["password"]

if not username:
    username = input("ArcGIS Enterprise username: ")
if not password:
    import getpass
    password = getpass.getpass("ArcGIS Enterprise password: ")

gis = GIS(
    url=enterprise["portal_url"],
    username=username,
    password=password,
    #verify_cert=False # Include if you are using self-signed certificates and are encountering an SSL error
    # Note: using this setting is a security risk.
)
print(f"Connected to ArcGIS Enterprise as: {gis.properties.user.username}")

## 4. Discover New STAC Items

Query the GeoCatalog for imagery matching the configured search parameters.

In [ ]:
# Build STAC search parameters
query = config["stac_query"]
search_params = StacSearchParameters(
    collections=query["collections"],
    limit=query["limit"],
)

if query.get("bbox"):
    search_params.bounding_box = query["bbox"]
if query.get("datetime"):
    search_params.date_time = query["datetime"]
if query.get("filter"):
    search_params.filter = query["filter"]
    if query.get("filter_lang"):
        search_params.filter_lang = query["filter_lang"]

# Execute search with pagination
source_asset_key = config["image_collection"]["source_asset_key"]
stac_items = []

print(f"Searching for: collections={query['collections']}, bbox={query.get('bbox')}, datetime={query.get('datetime')}")
print(f"Asset key: {source_asset_key}")
print()

response = mpc_client.stac.search(body=search_params)
page_num = 0
while True:
    features = response.get("features", [])
    if not features:
        break
    page_num += 1
    
    for feature in features:
        item_id = feature.get("id", "")
        collection = feature.get("collection", "")
        assets = feature.get("assets", {})
        
        # Extract the target asset
        asset = assets.get(source_asset_key)
        if asset is None:
            for key in ["data", "B04", "image"]:
                asset = assets.get(key)
                if asset:
                    break
        
        if asset:
            stac_items.append({
                "item_id": item_id,
                "collection": collection,
                "datetime": feature.get("properties", {}).get("datetime"),
                "href": asset.get("href", ""),
            })
    
    # Check for next page
    links = response.get("links", [])
    next_link = next((l for l in links if l.get("rel") == "next"), None)
    if next_link and next_link.get("body"):
        response = mpc_client.stac.search(body=next_link["body"])
    else:
        break

print(f"Discovered {len(stac_items)} STAC items across {page_num} page(s)")
print()

# Show first few items
for item in stac_items[:5]:
    print(f"  {item['item_id']} ({item['collection']}) — {item['datetime']}")
if len(stac_items) > 5:
    print(f"  ... and {len(stac_items) - 5} more")

## 5. Check Image Collection for Existing Items

Query the ArcGIS Image Collection to find items already ingested,
then compute the set of truly new items to add.

In [ ]:
# Helper: map blob URL to cloud store path
def blob_url_to_cloud_path(blob_url: str) -> str:
    """Convert a full blob URL to a cloud data store relative path."""
    cs = config["cloud_store"]
    parsed = urlparse(blob_url)
    path_parts = parsed.path.lstrip("/").split("/", 1)
    if len(path_parts) < 2:
        return blob_url
    relative_path = path_parts[1]
    return f"/cloudStores/{cs['store_name']}/{relative_path}"


# Search for existing image collection
collection_name = config["image_collection"]["name"]
results = gis.content.search(
    query=f'title:"{collection_name}" AND type:"Image Collection"',
    max_items=10,
)
collection_item = None
for item in results:
    if item.title == collection_name:
        collection_item = item
        break

existing_ids = set()
if collection_item:
    print(f"Found existing image collection: {collection_item.title} (id={collection_item.id})")
    
    # Query existing items for deduplication
    try:
        imagery_layer = ImageryLayer(collection_item.url, gis=gis)
        query_result = imagery_layer.query(
            where="1=1",
            out_fields="Name",
            return_geometry=False,
        )
        for feature in query_result.features:
            name = feature.attributes.get("Name", "")
            if name:
                existing_ids.add(name)
        print(f"  Already contains {len(existing_ids)} items")
    except Exception as e:
        print(f"  Could not query existing items: {e}")
else:
    print(f"Image collection '{collection_name}' not found — will be created")

# Filter to new items only
new_items = [item for item in stac_items if item["item_id"] not in existing_ids]
print(f"\nNew items to add: {len(new_items)} (skipping {len(stac_items) - len(new_items)} already in collection)")

## 6. Add New Rasters to Image Collection

Map the STAC item blob URLs to cloud store paths and add them to the
image collection. ArcGIS Image Server accesses the COGs directly from
blob storage via the registered cloud data store.

In [ ]:
if len(new_items) == 0:
    print("No new items to add. Image collection is up to date.")
else:
    # Map blob URLs to cloud store paths
    cloud_paths = [blob_url_to_cloud_path(item["href"]) for item in new_items]
    item_ids = [item["item_id"] for item in new_items]
    
    print(f"Adding {len(new_items)} rasters to image collection...")
    print(f"  Sample cloud store path: {cloud_paths[0]}")
    print()
    
    ic_config = config["image_collection"]
    
    if collection_item is None:
        # Create new image collection with initial rasters
        print("Creating new image collection...")
        collection_item = create_image_collection(
            image_collection=ic_config["name"],
            input_rasters=cloud_paths,
            raster_type_name=ic_config["raster_type_name"],
            gis=gis,
            context={
                "description": ic_config["description"],
                "outSR": {"wkid": ic_config["coordinate_system"]},
            },
        )
        print(f"Image collection created: {collection_item.title}")
    else:
        # Add rasters to existing collection
        imagery_layer = ImageryLayer(collection_item.url, gis=gis)
        raster_items = [{"uri": path, "name": sid} for path, sid in zip(cloud_paths, item_ids)]
        result = imagery_layer.add_rasters(
            raster_type=ic_config["raster_type_name"],
            raster_items=raster_items,
        )
        print(f"Added {len(cloud_paths)} rasters to existing collection")
        print(f"  Result: {result}")

## 7. Publish / Refresh Imagery Layer

Ensure the image collection is published as an imagery layer on Image Server.
If the service exists, refresh it to pick up the new rasters.

In [ ]:
if len(new_items) == 0:
    print("No changes to publish.")
else:
    # Check for existing imagery layer
    service_results = gis.content.search(
        query=f'title:"{collection_name}" AND type:"Imagery Layer"',
        max_items=10,
    )
    service_item = None
    for item in service_results:
        if item.title == collection_name:
            service_item = item
            break
    
    if service_item:
        # Refresh existing service
        print(f"Refreshing existing imagery layer: {service_item.title}")
        imagery_layer = ImageryLayer(service_item.url, gis=gis)
        imagery_layer.refresh_service()
        print(f"  Service refreshed: {service_item.url}")
    else:
        # Publish new service
        print(f"Publishing new imagery layer from collection...")
        service_item = collection_item.publish()
        print(f"  Published: {service_item.title}")
        print(f"  URL: {service_item.url}")

## 8. Summary

In [ ]:
print("="*60)
print("  Publishing Automation — Summary")
print("="*60)
print(f"  Run time:          {datetime.utcnow().isoformat()}Z")
print(f"  GeoCatalog:        {config['geocatalog']['endpoint']}")
print(f"  Collections:       {config['stac_query']['collections']}")
print(f"  Items discovered:  {len(stac_items)}")
print(f"  Items new:         {len(new_items)}")
print(f"  Items skipped:     {len(stac_items) - len(new_items)}")
print(f"  Image collection:  {collection_name}")
if len(new_items) > 0:
    print(f"  Status:            Updated")
else:
    print(f"  Status:            Up to date")
print("="*60)